In [1]:
# ==========================================
# Import Libraries
# ==========================================
import pandas as pd
import numpy as np

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import PCA

import matplotlib.pyplot as plt
import seaborn as sns

# ==========================================
# Load Dataset
# ==========================================
df = pd.read_csv("pagerank_scores.csv")

print(df.head())
print(df.info())

# ==========================================
# K-MEANS CLUSTERING ON PAGERANK SCORES
# ==========================================

# Assume dataset contains:
# page_id | pagerank_score

X = df[['pagerank_score']]

# Standardize values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Apply K-Means
kmeans = KMeans(
    n_clusters=3,
    random_state=42,
    n_init=10
)

df['cluster'] = kmeans.fit_predict(X_scaled)

print(df[['pagerank_score', 'cluster']].head())

# ==========================================
# Visualize Clusters
# ==========================================

plt.figure(figsize=(8,5))

sns.scatterplot(
    x=df.index,
    y='pagerank_score',
    hue='cluster',
    palette='viridis',
    data=df
)

plt.title("PageRank Clusters")
plt.xlabel("Page Index")
plt.ylabel("PageRank Score")
plt.show()

# ==========================================
# NLP ANALYSIS
# ==========================================

# Assume dataset contains a text column:
# content

if 'content' in df.columns:

    # Convert text to TF-IDF features
    vectorizer = TfidfVectorizer(
        stop_words='english',
        max_features=1000
    )

    tfidf_matrix = vectorizer.fit_transform(
        df['content'].fillna('')
    )

    print("TF-IDF Shape:", tfidf_matrix.shape)

    # ======================================
    # NLP + KMeans
    # ======================================

    text_kmeans = KMeans(
        n_clusters=3,
        random_state=42,
        n_init=10
    )

    text_clusters = text_kmeans.fit_predict(tfidf_matrix)

    df['text_cluster'] = text_clusters

    # ======================================
    # Reduce dimensions for plotting
    # ======================================

    pca = PCA(n_components=2)

    reduced = pca.fit_transform(
        tfidf_matrix.toarray()
    )

    plt.figure(figsize=(10,6))

    sns.scatterplot(
        x=reduced[:,0],
        y=reduced[:,1],
        hue=df['text_cluster'],
        palette='Set2'
    )

    plt.title("NLP Text Clusters")
    plt.xlabel("PCA Component 1")
    plt.ylabel("PCA Component 2")
    plt.show()

# ==========================================
# Cluster Statistics
# ==========================================

print("\nAverage PageRank by Cluster:\n")

print(
    df.groupby('cluster')['pagerank_score']
      .mean()
      .sort_values(ascending=False)
)

# ==========================================
# Save Results
# ==========================================

df.to_csv(
    "pagerank_clustered_results.csv",
    index=False
)

print("Results saved to pagerank_clustered_results.csv")

    node_1         node_2    score
0  DB01620        DB01620  0.15897
1  DB01620  ENTREZ:510296  0.07913
2  DB01620  ENTREZ:458542  0.07542
3  DB01620        DB02709  0.00927
4  DB01620        DB00366  0.00902
<class 'pandas.DataFrame'>
RangeIndex: 54110 entries, 0 to 54109
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   node_1  54110 non-null  str    
 1   node_2  54110 non-null  str    
 2   score   54110 non-null  float64
dtypes: float64(1), str(2)
memory usage: 2.1 MB
None


KeyError: "None of [Index(['pagerank_score'], dtype='str')] are in the [columns]"